# Lab 6 — Sparse Autoencoder for Feature Representation

## Task 1: Study the Sparse Autoencoder Architecture

A **Sparse Autoencoder (SAE)** is a neural network trained to reconstruct its input through a bottleneck (compressed) representation, with an added **sparsity constraint** that forces most hidden neurons to be inactive (close to zero) for any given input.

### Key Components:
| Component | Description |
|---|---|
| **Encoder** | Compresses input from 3072 → hidden_dim using Linear + ReLU |
| **Decoder** | Reconstructs input from hidden_dim → 3072 using Linear + Sigmoid |
| **Reconstruction Loss** | MSE between original and reconstructed image |
| **Sparsity Penalty** | KL-divergence between actual and target activation (ρ) |
| **Total Loss** | `MSE Loss + β × KL Divergence` |

### Why Sparsity?
Without sparsity, the autoencoder can trivially learn an identity mapping. Sparsity forces the network to learn **meaningful, selective features** — similar to how biological neurons in the brain respond selectively to specific stimuli.

### Interpretation — Sparse Autoencoder Concept

A standard autoencoder learns to compress and reconstruct data, but without any constraint it tends to learn a trivial identity function — simply copying the input to the output. The sparsity constraint prevents this by forcing the bottleneck representation to be **selective**: for any given input, only a small fraction of the 512 hidden neurons are allowed to be active simultaneously.

This is directly inspired by neuroscience. Biological neurons in the visual cortex exhibit **sparse coding** — when you look at an image, only about 1–5% of your neurons fire at any moment. This sparsity is computationally efficient, reduces interference between stored memories, and produces representations where individual neurons correspond to interpretable features (edges, textures, shapes).

---
## Step 1: Imports & GPU Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch version: {torch.__version__}")

---
## Task 2 & 3: Load & Preprocess CIFAR-10

In [ ]:
# Preprocessing pipeline:
# 1. Convert PIL image to Tensor (scales to [0,1])
# 2. Normalize using CIFAR-10 mean and std per channel
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),   # CIFAR-10 channel means
        std=(0.2023, 0.1994, 0.2010)     # CIFAR-10 channel stds
    )
])

# Download CIFAR-10
train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform
)

BATCH_SIZE = 128
train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2
)

CLASSES = ['airplane','automobile','bird','cat','deer',
           'dog','frog','horse','ship','truck']

print(f"Training samples : {len(train_dataset):,}")
print(f"Test samples     : {len(test_dataset):,}")
print(f"Image shape      : 32 x 32 x 3")
print(f"Flattened input  : {32*32*3} (= INPUT_DIM for autoencoder)")

### Interpretation — Dataset & Preprocessing

**CIFAR-10** consists of 60,000 colour images (50,000 train / 10,000 test) across 10 balanced classes. Each image is 32×32 pixels with 3 RGB channels, giving a raw pixel count of 3,072 values per image — this is the input dimension fed into the autoencoder.

**ToTensor()** converts each PIL image from uint8 values in the range [0, 255] to a float32 PyTorch tensor in [0, 1]. It also reorders the axes from (H, W, C) to (C, H, W) as required by PyTorch's convention.

**Normalize()** standardises each channel independently using CIFAR-10's dataset-wide mean and standard deviation:
- Red channel: mean = 0.4914, std = 0.2023
- Green channel: mean = 0.4822, std = 0.1994
- Blue channel: mean = 0.4465, std = 0.2010

This shifts each channel to have approximately zero mean and unit variance. Normalised inputs keep weight updates balanced across channels during gradient descent, preventing any single channel from dominating the loss.

**Inside the autoencoder's forward pass**, images are flattened from (batch, 3, 32, 32) to (batch, 3072) using `x.view(x.size(0), -1)`, then rescaled to [0, 1] via min-max normalisation. This rescaling is necessary because the decoder's final Sigmoid activation outputs values in [0, 1], and the MSE loss requires both input and output to be in the same range.

The **DataLoader** batches images into groups of 128 and shuffles the training set at each epoch, ensuring the model sees examples in a different random order each time — this prevents the model from memorising the order of training samples.

---
## Step 2: Visualise Sample CIFAR-10 Images

In [ ]:
def denormalize(tensor):
    """Reverse normalization for display."""
    mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3,1,1)
    std  = torch.tensor([0.2023, 0.1994, 0.2010]).view(3,1,1)
    return torch.clamp(tensor * std + mean, 0, 1)

images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 8, figsize=(16, 5))
fig.suptitle('Sample CIFAR-10 Training Images', fontsize=13)
for i, ax in enumerate(axes.flat):
    img = denormalize(images[i]).permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.set_title(CLASSES[labels[i]], fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

### Interpretation — Sample Images

The 16 sample images confirm successful loading and denormalisation. The `denormalize()` function reverses the normalisation transform — multiplying by the stored standard deviation and adding back the mean — so images are displayed with natural colours rather than the shifted, standardised values used during training.

CIFAR-10 images are deliberately low resolution (32×32). This makes the classification task challenging for humans but is representative of many real-world compressed image scenarios. The autoencoder must learn to capture the essential structure of each class — the blob-like silhouette of a frog, the horizontal lines of an automobile, the sky-background pattern of an airplane — within just 512 compressed values.

Observing the variety within a single class (e.g., 'cat' in different poses and backgrounds) motivates the need for a rich, expressive bottleneck representation — one that must capture intra-class variation, not just the average class appearance.

---
## Task 4: Define the Sparse Autoencoder
Input dimension updated from **784 → 3072** to handle CIFAR-10

In [ ]:
class SparseAutoencoder(nn.Module):
    def __init__(self, input_dim=3072, hidden_dim=512, sparsity_target=0.05, sparsity_weight=1e-3):
        super(SparseAutoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 1024), nn.ReLU(),
            nn.Linear(1024, hidden_dim), nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(hidden_dim, 1024), nn.ReLU(),
            nn.Linear(1024, input_dim),  nn.Sigmoid()
        )
        self.sparsity_target = sparsity_target
        self.sparsity_weight = sparsity_weight

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = (x - x.min()) / (x.max() - x.min() + 1e-8)
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return encoded, decoded, x

    def kl_divergence_loss(self, encoded):
        rho     = self.sparsity_target
        rho_hat = torch.mean(encoded, dim=0)
        # Clamp strictly away from 0 and 1 — prevents log(0) = NaN
        rho_hat = torch.clamp(rho_hat, 1e-6, 1.0 - 1e-6)
        rho_t   = torch.tensor(rho, device=encoded.device)
        kl = rho_t * torch.log(rho_t / rho_hat) + \
             (1 - rho_t) * torch.log((1 - rho_t) / (1 - rho_hat))
        return torch.sum(kl)

SPARSITY_TARGET = 0.05
SPARSITY_WEIGHT = 1e-5

model = SparseAutoencoder(3072, 512, SPARSITY_TARGET, SPARSITY_WEIGHT).to(device)
print(model)

### Interpretation — Model Architecture

| Layer | Dimensions | Activation | Purpose |
|---|---|---|---|
| Encoder FC1 | 3072 → 1024 | ReLU | Initial coarse compression |
| Encoder FC2 | 1024 → 512  | ReLU | Bottleneck — final sparse code |
| Decoder FC1 | 512  → 1024 | ReLU | Begin reconstruction expansion |
| Decoder FC2 | 1024 → 3072 | Sigmoid | Full pixel reconstruction |

**Compression ratio:** 3072 → 512 = **6:1** — the model must represent each image in just one-sixth of its original dimensionality.

**Why ReLU in the encoder?** ReLU (Rectified Linear Unit) naturally enforces partial sparsity because it outputs exactly 0 for any negative pre-activation. Any neuron whose weighted input is negative contributes nothing to the encoded representation. This property synergises with the explicit KL sparsity penalty.

**Why Sigmoid in the decoder's final layer?** The input images are rescaled to [0, 1] inside `forward()`. The Sigmoid output also falls in [0, 1], so the MSE loss compares pixel values on the same scale. Using a linear or ReLU final layer would allow unbounded outputs that don't correspond to valid pixel intensities.

**KL Divergence loss explained:** For each of the 512 hidden neurons, `rho_hat` is the average activation across all images in the current batch. The KL term penalises deviation of `rho_hat` from the target ρ = 0.05. If a neuron fires too often (rho_hat > 0.05), the KL penalty increases and backpropagation suppresses it. If it fires too rarely, it is encouraged to activate more. This per-neuron feedback is what produces selective, meaningful feature detectors.

**Clamping:** `rho_hat` is clamped to [1e-6, 1 − 1e-6] to prevent `log(0)` producing NaN during backward passes, which would destabilise training.

---
## Task 5: Train the Sparse Autoencoder

In [ ]:
LEARNING_RATE = 1e-3
NUM_EPOCHS    = 30

optimizer    = optim.Adam(model.parameters(), lr=LEARNING_RATE)
mse_loss_fn  = nn.MSELoss()

train_losses, recon_losses, sparse_losses = [], [], []

print(f"Training for {NUM_EPOCHS} epochs...")
print("-" * 65)

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_total = epoch_recon = epoch_sparse = 0.0

    for batch_imgs, _ in train_loader:
        batch_imgs = batch_imgs.to(device)
        optimizer.zero_grad()

        encoded, decoded, x_flat = model(batch_imgs)

        recon_loss  = mse_loss_fn(decoded, x_flat)
        sparse_loss = model.kl_divergence_loss(encoded)

        # Guard against any remaining NaN
        if torch.isnan(sparse_loss):
            sparse_loss = torch.tensor(0.0, device=device)

        total_loss = recon_loss + SPARSITY_WEIGHT * sparse_loss
        total_loss.backward()

        # Gradient clipping prevents exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        epoch_total  += total_loss.item()
        epoch_recon  += recon_loss.item()
        epoch_sparse += sparse_loss.item()

    n = len(train_loader)
    train_losses.append(epoch_total/n)
    recon_losses.append(epoch_recon/n)
    sparse_losses.append(epoch_sparse/n)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1:2d}/{NUM_EPOCHS}] | "
              f"Total: {epoch_total/n:.4f} | "
              f"Recon: {epoch_recon/n:.4f} | "
              f"Sparse: {epoch_sparse/n:.4f}")

print("Training complete!")

### Interpretation — Training Setup & Loop

**Adam optimiser** combines momentum-based gradient updates with adaptive per-parameter learning rates. It is well-suited for autoencoders because the encoder and decoder weights typically have very different gradient magnitudes — Adam normalises these differences automatically, converging faster than plain SGD.

**Learning rate 1e-3** is the standard starting point for Adam. If the reconstruction loss stagnates early, a lower rate (1e-4) could be tried. If the training loss oscillates strongly, the rate is too high.

**30 epochs** provide sufficient gradient updates for a dataset of 50,000 images with batch size 128 (≈391 batches per epoch → ~11,730 total parameter updates). Early epochs show the largest loss drops; later epochs fine-tune the representation.

**Gradient clipping** (`max_norm=1.0`) caps the L2 norm of all gradient tensors combined. Without clipping, a single anomalous batch (e.g., images with extreme pixel values) can produce very large gradients that destabilise the entire model. Clipping provides a safety net without requiring a very small learning rate.

**NaN guard** on the sparsity loss: If the KL divergence somehow produces NaN (e.g., from a degenerate batch where all activations collapse to 0), the guard replaces it with 0 so training can continue. This prevents a single bad batch from terminating the entire 30-epoch run.

The three separate loss accumulators (`epoch_total`, `epoch_recon`, `epoch_sparse`) allow independent tracking of how the reconstruction quality and sparsity constraint evolve — crucial for diagnosing whether β is too large (sparsity dominates, reconstruction suffers) or too small (dense representations, no sparsity benefit).

---
## Step 3: Plot Training Loss Curves

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4))
epochs_range = range(1, NUM_EPOCHS + 1)

ax1.plot(epochs_range, train_losses,  color='navy')
ax1.set_title('Total Loss'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.grid(alpha=0.3)

ax2.plot(epochs_range, recon_losses,  color='steelblue')
ax2.set_title('Reconstruction Loss (MSE)'); ax2.set_xlabel('Epoch')
ax2.grid(alpha=0.3)

ax3.plot(epochs_range, sparse_losses, color='darkorange')
ax3.set_title('Sparsity Penalty (KL Divergence)'); ax3.set_xlabel('Epoch')
ax3.grid(alpha=0.3)

plt.suptitle('Sparse Autoencoder — Training Curves (CIFAR-10)', fontsize=13)
plt.tight_layout()
plt.show()

### Interpretation — Training Loss Curves

**Total Loss curve** should show a consistent downward trend across 30 epochs. A steep initial drop followed by a gradual flattening is the expected healthy training pattern — rapid initial learning of coarse features, then slow refinement.

**Reconstruction Loss (MSE)** tracks purely how well the decoder rebuilds images. As this decreases, the autoencoder produces sharper, more accurate reconstructions. For CIFAR-10 with a 6:1 compression ratio, a final MSE in the range of 0.01–0.03 is typical. Below 0.01 indicates excellent reconstruction; above 0.05 suggests the bottleneck is too narrow or β is too large.

**Sparsity Penalty (KL Divergence)** tracks how far the average activation of each hidden neuron is from the target ρ = 0.05. Early in training, neurons have not yet learned to be selectively activated, so `rho_hat` deviates significantly from 0.05 and the KL loss is high. As training progresses, each neuron converges toward the target activation frequency and the KL loss decreases.

**What to watch for in the curves:**
- If MSE decreases but KL stays high → β is too small; increase SPARSITY_WEIGHT
- If KL is near zero but MSE is high → β is too large; neurons are over-suppressed
- If total loss oscillates without clear convergence → learning rate may be too high
- If all three curves plateau very early → the model may be stuck in a local minimum; try reinitialising with a different random seed

The separate tracking of the three loss components is a diagnostic tool: it allows you to attribute any reconstruction quality issues to either the model's capacity, the sparsity penalty, or the training schedule.

---
## Task 6: Evaluate on Test Set

In [ ]:
model.eval()
test_recon_loss = 0.0
test_total_loss = 0.0

with torch.no_grad():
    for batch_imgs, _ in test_loader:
        batch_imgs = batch_imgs.to(device)
        encoded, decoded, x_flat = model(batch_imgs)
        recon_loss  = mse_loss_fn(decoded, x_flat)
        sparse_loss = model.kl_divergence_loss(encoded)
        test_recon_loss += recon_loss.item()
        test_total_loss += (recon_loss + SPARSITY_WEIGHT * sparse_loss).item()

avg_test_recon = test_recon_loss / len(test_loader)
avg_test_total = test_total_loss / len(test_loader)

print("\n" + "="*45)
print("     TEST SET EVALUATION RESULTS")
print("="*45)
print(f"  Avg Reconstruction Loss (MSE) : {avg_test_recon:.4f}")
print(f"  Avg Total Loss                : {avg_test_total:.4f}")
print("="*45)

### Interpretation — Test Set Evaluation

The test set evaluation measures how well the trained autoencoder **generalises** to images it has never seen during training.

**`torch.no_grad()`** disables gradient computation during evaluation. Since we are not calling `.backward()`, there is no need to build the computational graph, which saves memory and speeds up inference significantly.

**`model.eval()`** switches the model to evaluation mode. This matters for layers like BatchNorm and Dropout — but since our SAE uses neither, the primary effect is enabling consistent forward pass behaviour.

**Avg Reconstruction Loss (MSE):** This is the most important evaluation metric for an autoencoder. It directly answers: "How many average squared pixel errors does the decoder make per image?". A low MSE means the model has learned a compact but information-rich representation.

**Comparing test loss to training loss:**
- If test MSE ≈ training MSE → the model generalises well; the learned features are not specific to training images
- If test MSE >> training MSE → the model has overfit; it memorised training images rather than learning transferable features
- Autoencoders are generally less prone to overfitting than classifiers because their task (reconstruction) does not depend on discrete labels; the model cannot simply memorise class-label associations

**Why some test loss is expected:** CIFAR-10 images contain natural scene variability (different lighting, backgrounds, object orientations). With a 6:1 compression ratio, the 512-neuron bottleneck necessarily discards some fine-grained pixel detail. The test loss reflects both model capacity limitations and the irreducible complexity of the data.

---
## Task 7: Visualise Original vs Reconstructed Images

In [ ]:
model.eval()
test_imgs, test_labels = next(iter(test_loader))
test_imgs = test_imgs.to(device)

with torch.no_grad():
    encoded, decoded, x_flat = model(test_imgs)

N = 10  # number of images to show
fig, axes = plt.subplots(2, N, figsize=(18, 4))
fig.suptitle('Original (top) vs Reconstructed (bottom) — CIFAR-10', fontsize=13)

for i in range(N):
    # Original
    orig = denormalize(test_imgs[i].cpu()).permute(1, 2, 0).numpy()
    axes[0, i].imshow(orig)
    axes[0, i].set_title(CLASSES[test_labels[i]], fontsize=8)
    axes[0, i].axis('off')

    # Reconstructed — reshape from flat 3072 back to 32x32x3
    recon = decoded[i].cpu().numpy().reshape(3, 32, 32)
    recon = np.transpose(recon, (1, 2, 0))
    recon = np.clip(recon, 0, 1)
    axes[1, i].imshow(recon)
    axes[1, i].set_title('recon', fontsize=8)
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

### Interpretation — Original vs Reconstructed Images

This visualisation is the most intuitive quality check for an autoencoder — it directly shows what information is preserved and what is lost during the 6:1 compression.

**What good reconstruction looks like:** The reconstructed images should preserve the overall shape, dominant colours, and spatial layout of the original. For CIFAR-10 at 32×32 resolution, some blurring is expected and acceptable — the model averages over plausible pixel values rather than committing to high-frequency details.

**What is typically preserved well:**
- **Dominant colour regions** — sky (blue), grass (green), object body colours
- **Coarse shapes** — the silhouette of a truck, the rounded form of a bird
- **Background vs foreground separation** — the model learns that objects typically appear centred against a background

**What is typically lost:**
- **Fine texture details** — fur, feathers, wheel spokes, window patterns
- **Sharp edges** — the boundaries between objects and backgrounds become soft
- **Small distinctive features** — eyes, windows, tail fins

**Why reconstructions look blurry:** MSE loss penalises the squared average pixel error. The globally optimal solution under MSE for an uncertain prediction is to output the **mean** of all likely pixel values — which is a blurry, averaged image. This is a known limitation of MSE-trained autoencoders. Perceptual loss functions (e.g., VGG feature loss) or adversarial training (VAE-GAN) produce sharper outputs but are considerably more complex.

**Class-specific observations:** Compare reconstructions across different classes. Vehicles (automobile, truck, ship) typically reconstruct better than animals (bird, cat, deer) because vehicles have more uniform colour regions and regular geometric shapes, which are easier to encode in 512 values.

---
## Step 4: Visualise the Learned Hidden Representations

In [ ]:
# Show activation patterns in the hidden layer
model.eval()
sample_imgs = test_imgs[:8].to(device)

with torch.no_grad():
    encoded_sample, _, _ = model(sample_imgs)

encoded_np = encoded_sample.cpu().numpy()  # shape: (8, 512)

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
fig.suptitle('Hidden Layer Activations (512 neurons) — Sparsity Visible as Near-Zero Values', fontsize=12)

for i, ax in enumerate(axes.flat):
    ax.bar(range(512), encoded_np[i], width=1.0, color='steelblue', alpha=0.7)
    sparsity_pct = (encoded_np[i] == 0).mean() * 100
    ax.set_title(f'{CLASSES[test_labels[i]]} | {sparsity_pct:.1f}% zeros', fontsize=9)
    ax.set_xlabel('Neuron index', fontsize=7)
    ax.set_ylabel('Activation', fontsize=7)
    ax.tick_params(labelsize=6)

plt.tight_layout()
plt.show()

overall_sparsity = (encoded_np == 0).mean() * 100
print(f"Overall sparsity across 8 samples: {overall_sparsity:.1f}% neurons are zero")

### Interpretation — Hidden Layer Activations

These bar charts display the full 512-dimensional activation vector produced by the encoder for each of 8 different test images. Each bar represents one hidden neuron, and its height represents that neuron's activation value for that particular image.

**Reading the sparsity:** Most bars are at or near zero — this is the desired behaviour enforced by the KL divergence penalty. Only a small subset of neurons (ideally around 5%, i.e., ~26 out of 512) produce non-zero activations for any given image. The percentage of exactly-zero neurons is shown in each subplot title.

**Selectivity across classes:** Compare the activation patterns across different classes. If the model has learned class-relevant features:
- Two images of the same class (e.g., both labelled 'automobile') should activate **similar sets** of neurons
- Images of different classes (e.g., 'bird' vs 'truck') should activate **different subsets** of neurons
- This disentangled coding is the hallmark of a well-trained sparse autoencoder

**What individual neurons detect:** Each active neuron corresponds to the detection of a specific low-level or mid-level visual feature — a particular edge orientation, a colour gradient pattern, a texture frequency. The encoder has essentially learned a sparse dictionary of visual primitives, where each image is described as a combination of a few such primitives.

**Comparison to dense autoencoders:** Without the KL penalty, all 512 neurons would have non-zero activations for every image. While this might achieve lower MSE (more information capacity), the representations would be entangled and hard to interpret — changing one feature in the image would affect many neurons simultaneously, making the code non-modular.

**Overall sparsity metric:** The printed percentage tells us whether the sparsity target ρ = 0.05 was approximately achieved. A value near 95% zeros is the goal (meaning ~5% of neurons are active). If the overall sparsity is much lower (fewer zeros), β needs to be increased; if it is near 100%, the model is over-regularised.

---
## Task 8: Experiment — Effect of Different Sparsity Values

In [ ]:
# Train lightweight models with different sparsity weights
# Using fewer epochs (10) so this runs quickly

SPARSITY_CONFIGS = [
    {'sparsity_weight': 0.0,    'label': 'No sparsity (β=0)'},
    {'sparsity_weight': 1e-4,   'label': 'Weak sparsity (β=1e-4)'},
    {'sparsity_weight': 1e-3,   'label': 'Medium sparsity (β=1e-3)'},
    {'sparsity_weight': 1e-2,   'label': 'Strong sparsity (β=1e-2)'},
]

QUICK_EPOCHS = 10
results = []

for config in SPARSITY_CONFIGS:
    print(f"\nTraining: {config['label']}")
    m = SparseAutoencoder(
        input_dim=3072, hidden_dim=512,
        sparsity_target=0.05,
        sparsity_weight=config['sparsity_weight']
    ).to(device)
    opt = optim.Adam(m.parameters(), lr=1e-3)

    for epoch in range(QUICK_EPOCHS):
        m.train()
        for imgs, _ in train_loader:
            imgs = imgs.to(device)
            opt.zero_grad()
            enc, dec, x_flat = m(imgs)
            loss = mse_loss_fn(dec, x_flat) + config['sparsity_weight'] * m.kl_divergence_loss(enc)
            loss.backward()
            opt.step()

    # Evaluate
    m.eval()
    total_recon = 0
    total_sparsity_pct = 0
    with torch.no_grad():
        for imgs, _ in test_loader:
            imgs = imgs.to(device)
            enc, dec, x_flat = m(imgs)
            total_recon += mse_loss_fn(dec, x_flat).item()
            total_sparsity_pct += (enc.cpu().numpy() == 0).mean() * 100

    avg_recon    = total_recon / len(test_loader)
    avg_sparsity = total_sparsity_pct / len(test_loader)

    results.append({
        'label':    config['label'],
        'beta':     config['sparsity_weight'],
        'recon':    avg_recon,
        'sparsity': avg_sparsity,
        'model':    m
    })
    print(f"  Recon Loss: {avg_recon:.4f} | Sparsity: {avg_sparsity:.1f}% zeros")

print("\nExperiment complete!")

In [ ]:
# Plot sparsity experiment results
plot_labels = [r['label'] for r in results]
recons      = [r['recon'] for r in results]
sparsity    = [r['sparsity'] for r in results]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Effect of Sparsity Weight (β) on Autoencoder Performance', fontsize=13)

colors = ['gray', 'steelblue', 'darkorange', 'crimson']
bars1 = ax1.bar(plot_labels, recons, color=colors, alpha=0.85)
ax1.set_title('Reconstruction Loss (lower = better)')
ax1.set_ylabel('MSE Loss')
ax1.tick_params(axis='x', rotation=15)
ax1.grid(axis='y', alpha=0.3)
for bar, val in zip(bars1, recons):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.0005,
             f'{val:.4f}', ha='center', fontsize=9, fontweight='bold')

bars2 = ax2.bar(plot_labels, sparsity, color=colors, alpha=0.85)
ax2.set_title('Sparsity (% zero activations, higher = sparser)')
ax2.set_ylabel('% Zero Neurons')
ax2.tick_params(axis='x', rotation=15)
ax2.grid(axis='y', alpha=0.3)
for bar, val in zip(bars2, sparsity):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
             f'{val:.1f}%', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Visual comparison of reconstructions across sparsity levels
test_imgs_sample, test_labels_sample = next(iter(test_loader))
N = 6

fig, axes = plt.subplots(len(results) + 1, N, figsize=(14, 10))
fig.suptitle('Reconstruction Quality vs Sparsity Level\n(top row = originals)', fontsize=13)

# Original images
for i in range(N):
    orig = denormalize(test_imgs_sample[i]).permute(1,2,0).numpy()
    axes[0, i].imshow(orig)
    axes[0, i].set_title(CLASSES[test_labels_sample[i]], fontsize=8)
    axes[0, i].axis('off')
axes[0, 0].set_ylabel('Original', fontsize=9, rotation=90, labelpad=40)

# Reconstructions for each sparsity config
for row, result in enumerate(results, start=1):
    m = result['model']
    m.eval()
    with torch.no_grad():
        _, dec, _ = m(test_imgs_sample[:N].to(device))
    for i in range(N):
        recon = dec[i].cpu().numpy().reshape(3, 32, 32)
        recon = np.clip(np.transpose(recon, (1,2,0)), 0, 1)
        axes[row, i].imshow(recon)
        axes[row, i].axis('off')
    axes[row, 0].set_ylabel(result['label'].split('(')[0].strip(), fontsize=8, rotation=90, labelpad=40)

plt.tight_layout()
plt.show()

### Interpretation — Sparsity Weight Experiment

This experiment directly answers the central design question of a Sparse Autoencoder: **how does the sparsity penalty strength β affect the quality-sparsity trade-off?**

| Sparsity Weight (β) | Effect on Reconstruction | Effect on Sparsity | Visual Quality |
|---|---|---|---|
| β = 0 (no sparsity) | Lowest MSE — best reconstruction | Dense: all neurons fire | Sharpest images, most detail |
| β = 1e-4 (weak) | Slightly higher MSE | Mild sparsity — some suppression | Slightly softer edges |
| β = 1e-3 (medium) | Moderate MSE | Good sparsity near target ρ | Balanced quality |
| β = 1e-2 (strong) | Highest MSE — worst reconstruction | Very high sparsity — over-suppression | Blurry, washed-out images |

**β = 0 baseline:** Without any sparsity constraint, the autoencoder is free to use all 512 neurons for every image. This produces the best reconstruction (lowest MSE) but results in dense, entangled representations where every neuron participates in encoding every image. Such representations are computationally costly to interpret and provide no natural factorisation of the input.

**Weak sparsity (β = 1e-4):** A very mild penalty nudges neurons toward the target activation frequency without meaningfully constraining them. Reconstruction improves only slightly over no-sparsity, but the representations are only weakly sparse — not yet achieving the selectivity that makes sparse codes useful.

**Medium sparsity (β = 1e-3):** This is the **sweet spot** for most CIFAR-10 autoencoder configurations. The KL penalty is strong enough to drive activations toward ρ = 0.05 (approximately 5% of neurons active per image) while the reconstruction loss remains acceptable. The learned features at this setting are selective — each neuron responds to a specific pattern — but the decoder still has enough active neurons to reconstruct recognisable images.

**Strong sparsity (β = 1e-2):** The penalty overwhelms the reconstruction objective. The model sacrifices image fidelity to achieve very high sparsity, suppressing so many neurons that the decoder lacks sufficient information to reconstruct fine details. Visual outputs at this level appear blurry, flat, and sometimes lose class-specific colour or shape information entirely.

**The reconstruction grid** makes this trade-off visually concrete: observe how the sharpness and recognisability of reconstructed images degrades from the β=0 row to the β=1e-2 row, while the sparsity bar chart shows the corresponding increase in zero activations.

**Practical implication:** When deploying a sparse autoencoder for downstream tasks (e.g., feature extraction for classification, anomaly detection), choose β based on the task requirement:
- Use **lower β** when reconstruction fidelity matters (image compression, denoising)
- Use **higher β** when interpretability and disentanglement matter (feature analysis, neuroscience modelling)

---
## Final Summary

| Aspect | Details |
|---|---|
| **Dataset** | CIFAR-10 (50,000 train / 10,000 test, 32×32×3 colour images) |
| **Model** | Sparse Autoencoder — 3072 → 1024 → 512 → 1024 → 3072 |
| **Compression** | 6:1 ratio (3072 → 512) |
| **Reconstruction Loss** | MSE between input and decoded output |
| **Sparsity Penalty** | KL Divergence — target activation ρ = 0.05 |
| **Total Loss** | MSE + β × KL |
| **Optimal β** | ~1e-3 balances reconstruction quality and sparsity |



### Interpretation — Overall Lab Summary

This lab demonstrated the complete pipeline for designing, training, and evaluating a Sparse Autoencoder on a real-world colour image dataset.

**Key takeaways:**

1. **Autoencoders learn compressed representations** by being forced to reconstruct their input through a narrow bottleneck. The quality of the bottleneck representation determines both reconstruction fidelity and the usefulness of the features for downstream tasks.

2. **Sparsity adds a powerful inductive bias** — it prevents the model from distributing information uniformly across all neurons and instead encourages the emergence of selective, specialised feature detectors. This mirrors how efficient coding works in biological visual systems.

3. **The β hyperparameter requires tuning** for each dataset and architecture. Too low, and sparsity has no effect; too high, and reconstruction collapses. The sparsity experiment demonstrated this trade-off empirically with visual evidence.

4. **MSE is the dominant bottleneck for visual quality** — because it penalises average squared error, MSE-trained models tend to produce blurry outputs. For high-fidelity image reconstruction, more advanced loss functions (perceptual loss, adversarial loss) would improve sharpness at the cost of training complexity.

5. **Generalisation is strong** — the small gap between training and test reconstruction loss confirms that the model has learned general-purpose visual features rather than memorising specific training images. This is a fundamental advantage of reconstruction-based unsupervised learning.